In [1]:
# Pre-training SoRL on arithmatic generalization dataset
# ------------------------------------------------------ 
import torch
from sorl.gat_sim import GAT, GATConfig
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup

BOS_TOKEN_ID = 20
gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 16],  # 16 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

Generate multiplication data: 

```python data/arithmetic.py```

In [4]:
from sorl.arithmetic import DigitTokenizer, ArithmeticDataLoader

tokenizer = DigitTokenizer(BOS_TOKEN_ID)
device = model.device
train_loader = ArithmeticDataLoader(min_digits=1, max_digits=1, num_examples=1000, pad_digits=2, device=device)
val_loader = ArithmeticDataLoader(min_digits=1, max_digits=1, num_examples=100, pad_digits=2, device=device)

100%|██████████| 100/100 [00:00<00:00, 93706.52it/s]


In [5]:
from sorl.neo_utils import sorl_search, sorl_evaluate
from sorl.info import SoRLLoss

# No abstract token ver. | Baseline
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

batch_size = 4
memory_span = 1792
attn_blocksize = 1792
num_steps = 500

for step in range(num_steps): 
 
    optimizer.zero_grad()

    tokens, doc_ids = train_loader.get_batch(batch_size)

    # --- compute loss ---
    base_traj_loss = model.forward(tokens, memory_span, attn_blocksize)[0].mean()
    loss = base_traj_loss
                                       
    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            val_tokens, _ = val_loader.get_batch(batch_size)
            val_loss = model.forward(val_tokens, memory_span, attn_blocksize)[0].mean() 
        print(f"validation step {step} | val_loss: {val_loss.item():.2f}")

validation step 0 | val_loss: 3.53
validation step 2 | val_loss: 3.33
validation step 4 | val_loss: 3.11
validation step 6 | val_loss: 2.91
validation step 8 | val_loss: 2.64
validation step 10 | val_loss: 2.49
validation step 12 | val_loss: 2.27
validation step 14 | val_loss: 2.13
validation step 16 | val_loss: 2.10
validation step 18 | val_loss: 1.79
validation step 20 | val_loss: 1.74
validation step 22 | val_loss: 1.68
validation step 24 | val_loss: 1.61
validation step 26 | val_loss: 1.70
validation step 28 | val_loss: 1.47
validation step 30 | val_loss: 1.44
validation step 32 | val_loss: 1.34
validation step 34 | val_loss: 1.24
validation step 36 | val_loss: 1.30
validation step 38 | val_loss: 1.17
validation step 40 | val_loss: 1.14
validation step 42 | val_loss: 1.13
validation step 44 | val_loss: 1.09
validation step 46 | val_loss: 1.07
validation step 48 | val_loss: 0.98
validation step 50 | val_loss: 1.05
validation step 52 | val_loss: 1.00
validation step 54 | val_loss: 0.

In [9]:
from sorl.neo_utils import sorl_search, sorl_evaluate
from sorl.info import SoRLLoss
from sorl.gapt import GatedPhaseTransition

# With Abstract token | SoRL

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

batch_size = 4
memory_span = 1792
attn_blocksize = 1792
K = 3  # abstraction ratio
n = 2  # number of rollout
temperatures = torch.tensor([0.0, 5.0], device=model.device)
max_iterations = 2  # of denoising steps for generating abstraction token (in parallel)
alpha_abs = 0.1 # weight on abstraction perplexity (training)
alpha_info_gain = 10.0
alpha_soft_zipf = 1.0
n_eval = 4  # number of rollout for evaluation
temperatures_eval = torch.tensor([0.0, 5.0, 5.0, 5.0], device=model.device) # temperature for evaluation
loss_fn = SoRLLoss(model.vocab_sizes[1])

num_steps = 500
gapt = GatedPhaseTransition(p_m=10)

for step in range(num_steps): 
 
    optimizer.zero_grad()

    tokens, doc_ids = train_loader.get_batch(batch_size)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    with torch.no_grad(): 
        search_tokens, search_ppt, search_adv = sorl_search(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures,
                                                               truncate_seq_len=False)
    
    # --- compute loss ---
    base_traj_loss = model.forward(tokens, memory_span, attn_blocksize)[0].mean()
    info_gain_loss, abs_loss, zipf_bigram_loss = loss_fn(search_tokens, model, base_traj_loss.detach(), memory_span, attn_blocksize)
    loss = base_traj_loss + alpha_info_gain * info_gain_loss + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss

    # --- log relative info gain ---
    rel_info_gain = info_gain_loss / base_traj_loss
                                       
    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            val_tokens, val_adv, traj_loss, abs_loss = sorl_evaluate(search_tokens, model, n=n_eval, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                   truncate_seq_len=False)
            # traj_loss, abs_loss = compute_loss(search_tokens, model, memory_span=memory_span, attn_blocksize=attn_blocksize)
            # vocab_util = compute_vocab_utilization_rate(val_tokens, model)
            vocab_util = 0.0
        print(f"validation step {step} | traj_loss: {traj_loss.item():.2f} | abs_loss: {abs_loss.item():.2f} | search adv: {val_adv.item() * 100:.2f}% | vocab util: {vocab_util * 100:.2f}%")

validation step 0 | traj_loss: 2.62 | abs_loss: 9.36 | search adv: -181.57% | vocab util: 0.00%
validation step 2 | traj_loss: 1.62 | abs_loss: 9.26 | search adv: 11.24% | vocab util: 0.00%
validation step 4 | traj_loss: 1.36 | abs_loss: 8.82 | search adv: 26.40% | vocab util: 0.00%
validation step 6 | traj_loss: 1.18 | abs_loss: 8.63 | search adv: 21.20% | vocab util: 0.00%
validation step 8 | traj_loss: 1.03 | abs_loss: 8.77 | search adv: 19.16% | vocab util: 0.00%
validation step 10 | traj_loss: 0.79 | abs_loss: 8.62 | search adv: 26.74% | vocab util: 0.00%
validation step 12 | traj_loss: 0.83 | abs_loss: 8.26 | search adv: 24.26% | vocab util: 0.00%
validation step 14 | traj_loss: 0.75 | abs_loss: 8.23 | search adv: 28.03% | vocab util: 0.00%
validation step 16 | traj_loss: 0.78 | abs_loss: 7.91 | search adv: -631.87% | vocab util: 0.00%
validation step 18 | traj_loss: 0.64 | abs_loss: 7.91 | search adv: 28.02% | vocab util: 0.00%
validation step 20 | traj_loss: 0.73 | abs_loss: 7.

In [12]:
# generate function implementation 
# ----------------------------------
from sorl.neo_utils import generate
from sorl.arithmetic import process_query, check_answer
from collections import defaultdict

K = 999
tokens, _ = val_loader.get_batch(1)
idx, answer_idx = process_query(tokens)

print(f"init   | idx: {idx[0].tolist()} | question: {tokenizer.decode(idx[0].tolist()[1:-1])}")
for i in range(len(answer_idx[0])*2): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, 
                   temperature=temperatures_eval[0])
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    # print(f"step {i+1} idx (abstraction free): {idx_without_abstraction.tolist()}")
    # print(f"                         idx : {idx[0].tolist()}")

is_correct, pred_answer, true_answer = check_answer(idx_without_abstraction, answer_idx, tokenizer)
print(f"is_correct: {is_correct} | pred_answer: {pred_answer} | true_answer: {true_answer}")

init   | idx: [20, 10, 14, 4, 2, 4, 10, 16, 4, 3] | question: 04 x 06 
is_correct: False | pred_answer: 12 | true_answer: 24
